# Bhaashik · NLLB adaptation on Colab / Kaggle

A complete **baseline → finetune → evaluate → error review** workflow for Bhojpuri, Maithili and Magahi. Uses a standard HF encoder–decoder, not a chat LLM.

Select a GPU runtime (T4 or better), enable Internet in Kaggle, and run in order. Free GPU availability and memory are not guaranteed. NLLB weights are CC-BY-NC 4.0. Accept upstream terms when required. Never paste tokens into cells. Example sentences are software fixtures, not research gold.

No notebook has been run on a hosted GPU as part of this repository's offline validation. The local integration tests exercise the same training, export and prediction functions with a tiny random model.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path('/kaggle/working/BhaashikMTFinetuning' if Path('/kaggle/working').exists() else '/content/BhaashikMTFinetuning')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/bhaashik/BhaashikMTFinetuning.git',str(ROOT)], check=True)
os.chdir(ROOT)

try:
    # Run the pip command and capture output to inspect failures
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', '.[train]'],
        capture_output=True,
        text=True,
        check=True
    )
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("--- STDOUT ---")
    print(e.stdout)
    print("--- STDERR ---")
    print(e.stderr)
    raise e

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'Choose a GPU runtime before training the 600M checkpoint.'
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
a = torch.randn(128, 128, device='cuda', dtype=torch.float16)
assert torch.isfinite(a @ a).all().item(), 'GPU FP16 smoke test failed'
del a
print('Free GPU memory (GiB):', round(torch.cuda.mem_get_info()[0]/2**30, 2))

--- STDOUT ---
Obtaining file:///content/BhaashikMTFinetuning
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached sacrebleu-2.4.3-py3-none-any.whl.metadata (51 kB)
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached datasets-3.1.0-py3-none-any.whl.metadata (20 kB)
  Using cached accelerate-1.1.1-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.13.2-py3-none-any.whl.metadata (13 kB)
  Using cached sentencepiece-0.2.0.tar.gz (2.6 MB)
  Installing build dependencies: started
  Installing build depend

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-e', '.[train]']' returned non-zero exit status 1.

## Choose a direction and data

Bhojpuri = `bho_Deva`, Maithili = `mai_Deva`, Magahi = `mag_Deva`, Hindi = `hin_Deva`, English = `eng_Latn`. All three focus languages are supported by NLLB already. Use a genuinely unsupported variety only after explicit extension.

For paper runs set `USE_EXAMPLE_DATA=False`, upload your licensed, document-disjoint JSONL splits to `data/local/{PAIR}/`, choose an immutable model commit SHA, and use a fresh run name per seed/condition. Freeze test data before any tuning. Kaggle input datasets can be copied into that local folder. Each row needs `id`, `source`, `target`; preserve `document_id`, `domain`, provenance and license.

In [ ]:
import json
SRC, TGT = 'bho_Deva', 'hin_Deva'
SEED, MODE = 42, 'lora'  # Repeat with seeds 13, 42, 87; compare full separately.
MODEL = 'facebook/nllb-200-distilled-600M'
REVISION = None  # Set an immutable Hub commit for final experiments.
USE_EXAMPLE_DATA = True  # Only a mechanics demonstration, not a research result.
PAIR = f"{SRC.split('_')[0]}-{TGT.split('_')[0]}"
if USE_EXAMPLE_DATA and (SRC, TGT) != ('bho_Deva', 'hin_Deva'):
    raise ValueError('Example data are Bhojpuri → Hindi only; supply matching data.')
DATA = Path('data/examples') if USE_EXAMPLE_DATA else Path('data/local') / PAIR
RUN = Path('runs') / f'{PAIR}-{MODE}-s{SEED}'
from bhaashik.data import audit
print(audit({s:DATA / f'{s}.jsonl' for s in ('train','dev','test')}))


## Untouched baseline

Use identical decoding and test rows for both systems. The test set is used only for this final comparison, not for selecting hyperparameters. This downloads roughly gigabytes of model artifacts; keep sufficient disk headroom. Public model download access is required.

In [ ]:
from bhaashik.modeling import translate
BASE = Path('runs') / f'{PAIR}-baseline'
translate(MODEL, SRC, TGT, DATA/'test.jsonl', BASE/'predictions.jsonl', revision=REVISION,
          batch_size=1, max_length=128, max_new_tokens=128, beams=4)


## Finetune and export a normal HF checkpoint

The default is LoRA with batch 1, gradient accumulation 16 and sequence length 128. This is a starting budget, not a guarantee of fitting every GPU. Full finetuning requires substantially more memory. If a runtime is interrupted, set `resume_from_checkpoint` to a saved `RUN/checkpoints/checkpoint-N` and retain the same data/config. Copy checkpoints to persistent storage regularly; free notebook disks are ephemeral.

In [ ]:
from bhaashik.modeling import train
config = dict(model=MODEL, revision=REVISION, src_lang=SRC, tgt_lang=TGT,
              train_file=str(DATA/'train.jsonl'), dev_file=str(DATA/'dev.jsonl'), output_dir=str(RUN),
              mode=MODE, seed=SEED, epochs=3, learning_rate=2e-4 if MODE=='lora' else 2e-5,
              batch_size=1, gradient_accumulation_steps=16, max_length=128,
              precision='fp16', gradient_checkpointing=True, lora_rank=8)
config_path = Path('configs') / f'{PAIR}-{MODE}-s{SEED}.json'
config_path.write_text(json.dumps(config, indent=2))
print(train(config_path))
(RUN / "training_config.json").write_text(json.dumps(config, indent=2))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# LoRA is merged; no PEFT wrapper/custom architecture is needed for consumers.
tokenizer = AutoTokenizer.from_pretrained(RUN)
model = AutoModelForSeq2SeqLM.from_pretrained(RUN)
print(type(model).__name__, len(tokenizer))
del model, tokenizer
import gc
gc.collect(); torch.cuda.empty_cache()
translate(RUN, SRC, TGT, DATA/'test.jsonl', RUN/'predictions.jsonl',
          batch_size=1, max_length=128, max_new_tokens=128, beams=4)


## Paired evaluation and human review

Corpus BLEU/chrF++ are not dialect-fidelity judgments. Bootstrap whole documents when IDs exist. A fixture-sized sample cannot support a paper claim; add qualified native-speaker annotation and multiple training seeds. Exported flags are triage hints, not automatic linguistic diagnoses. Import `evaluation.json` in the Pages portal to view actual scores.

In [ ]:
from bhaashik.evaluation import evaluate, export_errors
result = evaluate(RUN/'predictions.jsonl', RUN/'evaluation.json',
                  baseline=BASE/'predictions.jsonl', samples=1000, seed=SEED)
print(json.dumps(result, ensure_ascii=False, indent=2))
export_errors(RUN/'predictions.jsonl', RUN/'annotations.csv', seed=SEED)


## Preserve artifacts

The archive includes the merged model/tokenizer, run manifest, checkpoint state, config, baseline and candidate predictions, metric signatures and annotation sheet. Training data are not automatically archived: preserve licensed splits and provenance separately. Review the generated model card before sharing. Exporting artifacts is not publishing them to the Hub.

In [ ]:
import shutil
archive = shutil.make_archive(str(ROOT.parent / f'bhaashik-{PAIR}-{MODE}-s{SEED}-artifacts'), 'zip', root_dir='.', base_dir='runs')
print('Download or copy to persistent storage:', archive)
# Colab: from google.colab import files; files.download(archive)
# Kaggle: use the notebook Output files panel.
